# Day 1 · Session 3 — Prompt Engineering That Actually Works

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksuresh/fdp-llm-agentic-ai/blob/main/day1-llm-foundations/03_prompt_engineering.ipynb)

---

**Duration:** ~30 minutes  
**Goal:** Master 6 prompt engineering techniques and build a reusable teaching assistant.

> *Adapted from Ed Donner's LLM Engineering course (week1/day5). Expanded with faculty-specific patterns.*

In [ ]:
!pip install openai python-dotenv -q

In [ ]:
import os, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def ask(prompt, system=None, model="gpt-4o-mini", temperature=0.7, json_mode=False):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    kwargs = dict(model=model, messages=messages, temperature=temperature, max_tokens=600)
    if json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ Setup complete")

---

## Part 1 — The Evolution of a Prompt

Watch how the same topic transforms through 4 rounds of refinement.

In [ ]:
# Round 1: Vague prompt
print("=" * 60)
print("ROUND 1 — Vague")
print("=" * 60)
r1 = ask("Explain overfitting")
print(r1)

In [ ]:
# Round 2: Add persona + context + format
print("=" * 60)
print("ROUND 2 — Persona + Context + Format")
print("=" * 60)
r2 = ask(
    "Explain overfitting in machine learning.",
    system="""
You are a teaching assistant for a 3rd-year B.Tech CSE course on Machine Learning
at an Indian engineering college. Students have studied linear regression but
not yet neural networks. Use relatable Indian examples. Keep responses under 100 words.
"""
)
print(r2)

In [ ]:
# Round 3: Add few-shot examples to nail the format
print("=" * 60)
print("ROUND 3 — Few-shot examples for consistent format")
print("=" * 60)
r3 = ask("""
Explain a machine learning concept using this format:

CONCEPT: Bias-Variance Tradeoff
DEFINITION: When a model's error comes from two sources — bias (wrong assumptions)
and variance (sensitivity to training data noise).
ANALOGY: Like a cricket bowler who always bowls to the same spot (high bias,
low variance) vs one who bowls unpredictably (low bias, high variance).
EXAM QUESTION: Which type of error does regularisation primarily reduce?

Now explain this concept using the exact same format:
CONCEPT: Overfitting
""", temperature=0.3)
print(r3)

In [ ]:
# Round 4: Add Chain-of-Thought for reasoning quality
print("=" * 60)
print("ROUND 4 — Chain-of-Thought for analytical depth")
print("=" * 60)
r4 = ask("""
A student trained a neural network and got:
- Training accuracy: 99%
- Validation accuracy: 62%

Think step by step about what is happening and what the student should do.
Then give a clear recommendation in 2 sentences.
""", temperature=0.2)
print(r4)

---

## Part 2 — Zero-shot vs Few-shot

In [ ]:
# Zero-shot sentiment classification
reviews = [
    "The practical sessions were excellent but theory was rushed.",
    "Best workshop I've attended. Very relevant to industry.",
    "Too basic for AI/ML faculty. Expected more depth.",
]

print("ZERO-SHOT — no examples provided:")
print("-" * 50)
for review in reviews:
    result = ask(f'Classify the sentiment: "{review}"\nRespond with only: Positive, Negative, or Mixed')
    print(f"Review: {review[:50]}...")
    print(f"Label:  {result.strip()}\n")

In [ ]:
# Few-shot with a custom schema
print("\nFEW-SHOT — with examples showing exact format:")
print("-" * 50)

few_shot_prompt = """
Classify faculty feedback. Use exactly this format:
Sentiment: [Positive/Negative/Mixed]
Aspect: [Content/Delivery/Relevance/Logistics]
Action: [one specific improvement suggestion]

Example 1:
Feedback: "Examples were clear but pacing was too fast."
Sentiment: Mixed
Aspect: Delivery
Action: Slow down during code demonstrations.

Example 2:
Feedback: "Excellent content, very practical."
Sentiment: Positive
Aspect: Content
Action: Continue using hands-on exercises.

Now classify:
Feedback: "{review}"
"""

for review in reviews:
    result = ask(few_shot_prompt.format(review=review), temperature=0.1)
    print(f"Input: {review}")
    print(result.strip())
    print()

---

## Part 3 — Chain-of-Thought

In [ ]:
problem = """
A university has 3 departments: CSE, ECE, and Mechanical.
CSE has 240 students, ECE has 180, Mechanical has 150.
30% of CSE, 25% of ECE, and 20% of Mechanical students
are enrolled in an AI elective.
What percentage of total students are enrolled in the AI elective?
"""

print("WITHOUT Chain-of-Thought:")
print("-" * 50)
no_cot = ask(problem + "\nAnswer:", temperature=0)
print(no_cot)

print("\nWITH Chain-of-Thought:")
print("-" * 50)
cot = ask(problem + "\nThink step by step, then give the final answer.", temperature=0)
print(cot)

---

## Part 4 — Structured JSON Output

In [ ]:
# Extract structured data from free-text student submissions
submission = """
Student Name: Arjun Mehta
Roll Number: 21CSB047
Course: Machine Learning (CS601)

Q1 Answer: A neural network is a computational system inspired by biological neurons.
It consists of input, hidden and output layers connected by weights.
Backpropagation adjusts weights using the chain rule of calculus.

Q2 Answer: Overfitting happens when the model memorises training data instead of
learning general patterns. Solutions include dropout, regularisation, and more data.
"""

result = ask(
    f"""
Extract the following from this student submission and return as JSON:
{{
  "student": {{"name": "", "roll": "", "course": ""}},
  "answers": [
    {{"question_num": 1, "key_concepts": [], "word_count": 0, "quality": "Good/Average/Poor"}},
    {{"question_num": 2, "key_concepts": [], "word_count": 0, "quality": "Good/Average/Poor"}}
  ],
  "overall_feedback": ""
}}

Submission:
{submission}
""",
    json_mode=True,
    temperature=0
)

data = json.loads(result)
print(json.dumps(data, indent=2))

# Now use the structured data programmatically
print(f"\n📊 Student: {data['student']['name']} ({data['student']['roll']})")
for ans in data['answers']:
    print(f"   Q{ans['question_num']}: {ans['quality']} | Concepts: {', '.join(ans['key_concepts'][:3])}")

---

## Part 5 — The 4 Faculty Prompt Patterns

In [ ]:
# ── Pattern 1: THE EXPLAINER ─────────────────────────────────
def explainer(concept, level, subject, domain):
    return ask(f"""
You are a teacher for {level} students studying {subject}.
Explain '{concept}' using:
1. A one-sentence definition (no jargon)
2. A real-world analogy from {domain}
3. One worked example with numbers
4. One common misconception students have about this topic
""", temperature=0.6)

print("THE EXPLAINER:")
print(explainer(
    concept="gradient descent",
    level="2nd year B.Tech CSE",
    subject="Machine Learning",
    domain="everyday Indian life"
))

In [ ]:
# ── Pattern 2: THE EXAMINER ──────────────────────────────────
def examiner(topic, n, level, difficulty):
    return ask(f"""
Create {n} MCQ questions on '{topic}' for {level} students.
Difficulty: {difficulty}

For each question provide:
- The question
- Options A, B, C, D
- Correct answer letter
- Brief explanation of why it's correct

Vary the question types: recall, application, and analysis.
Do not repeat similar question patterns.
""", temperature=0.8)

print("THE EXAMINER:")
print(examiner(
    topic="Support Vector Machines",
    n=3,
    level="3rd year B.Tech CSE",
    difficulty="Medium"
))

In [ ]:
# ── Pattern 3: THE SOCRATIC GUIDE ────────────────────────────
def socratic_response(student_question, topic_context):
    return ask(
        student_question,
        system=f"""
You are a Socratic tutor for a {topic_context} course.
NEVER give direct answers. Instead:
1. Identify what the student already knows from their question
2. Ask ONE probing question that leads them toward the answer
3. If they mention a partial truth, build on it
4. Keep responses under 60 words
5. Use an encouraging, curious tone
""",
        temperature=0.7
    )

print("THE SOCRATIC GUIDE:")
questions = [
    "Why does my model perform great in training but badly in testing?",
    "I think it's because I have too many parameters. Is that right?",
    "So I should add dropout? But won't that make training accuracy worse?"
]
for q in questions:
    print(f"\n👤 Student: {q}")
    print(f"🤖 Tutor:   {socratic_response(q, 'Machine Learning')}")

In [ ]:
# ── Pattern 4: THE GRADER ─────────────────────────────────────
def grade_submission(submission_text, topic, rubric):
    result = ask(f"""
Grade this student submission on '{topic}'.

Rubric:
{rubric}

Provide your response as JSON:
{{
  "scores": {{"criterion": score}},
  "total": 0,
  "grade": "A/B/C/D/F",
  "strengths": ["...", "..."],
  "improvements": ["...", "..."],
  "encouraging_comment": "..."
}}

Submission:
{submission_text}
""", json_mode=True, temperature=0.2)
    return json.loads(result)

print("THE GRADER:")
feedback = grade_submission(
    submission_text="""Overfitting is when a model learns too much from training data.
    It happens when the model is too complex. You can fix it by using less data
    or making the model simpler. Regularization can also help.""",
    topic="Overfitting in Machine Learning",
    rubric="Correctness (40%), Clarity (30%), Depth (30%)"
)
print(json.dumps(feedback, indent=2))

---

## Part 6 — Prompt Iteration Practice

In [ ]:
# This cell shows prompt iteration — the most important skill
# Run it multiple times with different prompts and compare

YOUR_TOPIC = "Decision Trees"            # ← Change this
YOUR_SUBJECT = "Machine Learning"        # ← Change this  
YOUR_AUDIENCE = "3rd year B.Tech CSE"    # ← Change this

# Iteration 1: Basic
v1 = ask(f"Explain {YOUR_TOPIC}")

# Iteration 2: With context
v2 = ask(f"Explain {YOUR_TOPIC} to a {YOUR_AUDIENCE} student studying {YOUR_SUBJECT}")

# Iteration 3: With format + Indian context
v3 = ask(
    f"Explain {YOUR_TOPIC} in exactly 3 bullet points, each under 25 words, using an Indian real-world example.",
    system=f"You teach {YOUR_SUBJECT} to {YOUR_AUDIENCE} students at an Indian college."
)

for i, (label, v) in enumerate([("V1 Basic", v1), ("V2 +Context", v2), ("V3 +Format", v3)], 1):
    print(f"{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(v[:400] + ("..." if len(v) > 400 else ""))
    print()

---

## 🎯 Lab Assignment

**File:** `lab/lab3_prompt_engineering.ipynb`

### Task 1 — Prompt Evolution
Pick a concept from your subject. Write 4 versions of a prompt — each adding one element (persona, context, format, examples). Show all 4 outputs.

### Task 2 — Chain-of-Thought  
Design a multi-step problem from your subject. Run it with and without CoT. Compare accuracy and reasoning quality.

### Task 3 — The Examiner
Use the `examiner()` function to generate 5 MCQs on a topic you teach this semester. Check each question for accuracy.

### Task 4 — JSON Grader
Take a student paragraph from your subject. Use `grade_submission()` to grade it. Adjust the rubric to match your actual marking criteria.

### 🌟 Bonus — Build Your Own Pattern
Create a new reusable prompt function (e.g., `case_study_generator()`, `lab_manual_writer()`, `quiz_generator()`). Share it with the group.

```python
# Your code here
def your_pattern(...):
    return ask(f"""...""")
```